# Comparison to existing methods

## Required installs for R

The following packages need to be installed in addition to the environment described in the `README.md`.

If you want to do this manually, run the following: 
```R
# GSReg
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("GSReg")

if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("GSBenchMark")
```

```R
# maSigPro
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install("maSigPro")
```

In [ ]:
%%bash
# Install the above packages within this notebook. This might be very slow.
# Alternatively, open a terminal and run by hand via: `conda activate nedis; R`.
# You might have to run this several times.
Rscript -e '
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!require("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
}

BiocManager::install(c(
    "GSReg",
    "GSBenchMark",
    "maSigPro"
))
'

## Setup environment

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from nedis.parallelization import set_threads_for_external_libraries
set_threads_for_external_libraries(4)

In [ ]:
import sys
import logging
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import sklearn.cluster


In [ ]:
import nedis.data.synthetic
import nedis.visualization
import nedis.cluster.leidenalg

from nedis.cluster.leidenalg import WeightedLeidenClustering
from nedis.data.synthetic import make_correlation_data_mixed
from nedis.visualization import plot_cordis_cluster

## Config

In [ ]:
# randomness
random_state = 43
np.random.seed(random_state)

In [ ]:
output_dir = pathlib.Path("../_out/synthetic")
output_dir.mkdir(parents=True, exist_ok=True)

## Initialize R

In [ ]:
%load_ext rpy2.ipython

In [ ]:
# test if R works
%R X=c(1,4,5,7); sd(X); mean(X)

## Comparison to `maSigPro`

### Data

In [ ]:
n_samples = 100
correlations = np.linspace(0.1,.9, 5)
data = [
    make_correlation_data_mixed(
        [5,10,5,5], 
        correlations=np.array([
            [1-c,0,0,0],
            [0,c,0,0],
            [0,0,1-c,-(1-c)],
            [0,0,-(1-c),1-c]]), 
        n_noise_features=15, 
        n_samples=n_samples,
        random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))
labels = np.repeat([0,1,2,-1], [5,10,10,15])

# make last feature to be associated with time for sanity check
# X[:,-1] += y
# X[:,-2] += y
# X[:,-3] += y
# X[:,-4] += y
# X[:,-5] += y

In [ ]:
# check correlation with time
from scipy.stats import spearmanr
for i in range(X.shape[1]):
    r, p = spearmanr(X[:,i], y)
    if p < 0.05:
        print(i, r, p)

In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 4))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    cor = correlation_matrices[yy]
    nedis.visualization.visualize_feature_clusters(cor, ax=ax, heatmap_kwargs=dict(cmap="vlag", center=0, vmin=-1, vmax=1))

In [ ]:
coordinates = {}
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = sklearn.manifold.TSNE(n_components=2, random_state=random_state, init="pca", learning_rate=200)
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

In [ ]:
fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 4 * 2))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    ax.axis("off")
    ax.set_title(yy)
    plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)


fig.savefig(output_dir / "clustering_comparison_ref.png", bbox_inches="tight")
fig.savefig(output_dir / "clustering_comparison_ref.pdf", bbox_inches="tight")

### Apply `maSigPro`

In [ ]:
%%R 
library(maSigPro)

In [ ]:
%%R -i X -i y -i entities -i labels

# data
mag.data <- t(X)
rownames(mag.data) <- paste("feature", c(1:dim(mag.data)[1]), sep = "")
colnames(mag.data) <- paste("sample_t-", y + 1, "_e-", entities + 1, sep = "")

# mag.data[1:10,1:10]

In [ ]:
%%R
# edesign
mag.edesign <- cbind(y + 1, y + 1, rep(1, length(entities)))
colnames(mag.edesign) <- c("Time", "Replicates", "Groups")
rownames(mag.edesign) <- colnames(mag.data)

# mag.edesign

In [ ]:
%%R
design <- make.design.matrix(mag.edesign, degree = 2)

In [ ]:
%%R
fit <- p.vector(mag.data, design, Q = 0.5, MT.adjust = "BH", min.obs = 5)
fit$i

In [ ]:
%%R
# lowest p-value out of 
# print(names(fit))
print(length(fit$p.adjusted))
print(min(fit$p.adjusted))

**Result:** `MaSigPro` measures changing gene values between time points and conditions, while `NeDis` looks for changing correlation structures. This is illustrated by the synthetic example which exhibits strong correlation changes, which `NeDis` discovers in the data (see `03_synthetic.ipynb`), while `maSigPro` does not find any significant signals (features/genes).

In [ ]:
%%R
citation("maSigPro")

## Comparison to `EVA`

### Data

In [ ]:
n_samples = 100
correlations = np.linspace(0.1,.9, 2)
data = [
    make_correlation_data_mixed(
        [5,10,5,5], 
        correlations=np.array([
            [1-c,0,0,0],
            [0,c,0,0],
            [0,0,1-c,-(1-c)],
            [0,0,-(1-c),1-c]]), 
        n_noise_features=15, 
        n_samples=n_samples,
        random_state=random_state + i) 
    for i, c in enumerate(correlations)]

X = np.concatenate(data)
y = np.concatenate([np.repeat(i, d.shape[0]) for i, d in enumerate(data)])
entities = np.tile(np.arange(n_samples), len(correlations))
labels = np.repeat([0,1,2,-1], [5,10,10,15])

# make last feature to be associated with time for sanity check
# X[:,-1] += y
# X[:,-2] += y
# X[:,-3] += y
# X[:,-4] += y
# X[:,-5] += y

In [ ]:
# check correlation with time
from scipy.stats import spearmanr
for i in range(X.shape[1]):
    r, p = spearmanr(X[:,i], y)
    if p < 0.05:
        print(i, r, p)

In [ ]:
y_unique = np.unique(y)
correlation_matrices = {}
for yy in y_unique: 
    cor = np.corrcoef(X[y == yy,:], rowvar=False)
    correlation_matrices[yy] = cor

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(6 * len(data), 4))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    cor = correlation_matrices[yy]
    nedis.visualization.visualize_feature_clusters(cor, ax=ax, heatmap_kwargs=dict(cmap="vlag", center=0, vmin=-1, vmax=1))

In [ ]:
coordinates = {}
for yy, correlation_matrix in correlation_matrices.items(): 
    tsne = sklearn.manifold.TSNE(n_components=2, random_state=random_state, init="pca", learning_rate=200)
    coo = tsne.fit_transform(abs(correlation_matrix))
    coordinates[yy] = coo

In [ ]:
fig, axes = plt.subplots(1, len(y_unique), figsize=(4 * 2 * len(y_unique), 4 * 2))
for i, yy in enumerate(y_unique):
    ax = axes[i]
    ax.axis("off")
    ax.set_title(yy)
    plot_cordis_cluster(None, coordinates[y_unique[0]], correlation_matrices[yy], ax=ax)

### Apply NeDis

In [ ]:
from nedis.cordis.default \
    import DefaultCorrelationDisruptionFeatureTransformer \
    as DefaultCorrelationDisruptionTransformer

In [ ]:
# configure and fir correlation disruption transformer
disruption_transformer = DefaultCorrelationDisruptionTransformer(
    default_optimization_separation_score="auc",
    default_derive_features_aggregation="mean",
    select_clusters=3)
disruption_transformer.fit(X, y, groups=entities, subset_masks="y");

In [ ]:
# visualize discovered disrupted modules
from nedis.visualization import plot_cordis_cluster as plot_cluster
y_unique = np.unique(y)
print(y_unique)

for i_cluster, cluster in enumerate(disruption_transformer.selected_clusters_):
    
    fig, axes = plt.subplots(
        1, len(y_unique), 
        figsize=(4 * 1 * (len(y_unique)), 4 * 1), 
        dpi=300)
    
    # cluster visualization
    for i, yy in enumerate(y_unique):
        ax = axes[i]
        ax.axis("off")
        plot_cluster(
            cluster, 
            coordinates[0],
            correlation_matrices[yy], 
            correlation_threshold=0, 
            verbose=0,
            ax=ax)
    fig.suptitle(
        f"Cluster: Reference data={cluster['reference_label']}, Id={cluster['id']}")
    
    plt.show()

### Apply `EVA`

In [ ]:
pathways = {l: np.where(labels == l)[0] for l in np.unique(labels)}
pathways

In [ ]:
# %%R
# diracpathways

In [ ]:
%%R -i X -i y -i entities -i labels

pathways <- list()
for (l in unique(labels)) {
    features <- which(labels %in% l)
    if (l == -1) l <- "noise"
    pathways[[paste("pathway_", l, sep="")]] <- paste("feature", features, sep = "")
}
pathways

In [ ]:
%%R -i X -i y -i entities -i labels

# data
exprsdata <- t(X)
rownames(exprsdata) <- paste("feature", c(1:dim(mag.data)[1]), sep = "")
colnames(exprsdata) <- paste("sample_t-", y + 1, "_e-", entities + 1, sep = "")

exprsdata[1:10,1:10]

In [ ]:
%%R -i X -i y -i entities -i labels

# phenotype
phenotypes <- factor(y)
phenotypes

In [ ]:
%%R
library(GSReg)

In [ ]:
%%R

#Calculating the variance for the pathways
#Calculate how much it takes to calculate the statistics and their p-value for all path
VarAnKendallV = GSReg.GeneSets.EVA(
    geneexpres=exprsdata,
    pathways=pathways,
    phenotypes=as.factor(phenotypes))

pvalustat = sapply(VarAnKendallV,function(x) x$pvalue);

print("p-values")
print(pvalustat)

print("p-values (adjusted)")
print(pvalustat * 4)

**Result:** `EVA` (expression variation analysis) measures variance in pathways across conditions, while `NeDis` looks for changing correlation structures. This is illustrated by the synthetic example, which exhibits strong correlation changes, which `NeDis` can easily discover, while `maSigPro` does not find any significant pathways.

In [ ]:
%%R
citation("GSReg")